# generte postionnal encoding matrix PE 

In [2]:
from positional_enoding import generate_postionnal_encoding_matrix
import numpy as np
from torch.nn import Softmax
tokenised_texte = ["the" , "cat" , "is" , "so" ]

PE_matrix = generate_postionnal_encoding_matrix(tokenised_texte=tokenised_texte)
print(PE_matrix)
print(np.shape(PE_matrix))


[[0.0, 1.0, 0.0, 0.03162277660168379], [0.8414709848078965, 0.5403023058681398, 0.02660964896937897, 0.01708585911584281], [0.9092974268256817, -0.4161468365471424, 0.02875450939299445, -0.013159718445627706], [0.1411200080598672, -0.9899924966004454, 0.004462606488904997, -0.03130631155733909]]
(4, 4)


# Create the embeding Matrix Xe

In [3]:

np.random.default_rng(42)
d_model = 4
Xe = np.random.rand(d_model,d_model)
Wq = np.random.rand(d_model,d_model)
Wk = np.random.rand(d_model,d_model)
Wv = np.random.rand(d_model,d_model)
X_embeding = Xe + PE_matrix
print(X_embeding)


[[ 0.98857478  1.81905566  0.19239015  0.57065012]
 [ 1.84058775  1.36130789  0.10781135  0.50744783]
 [ 1.38178525 -0.27146208  1.01095954  0.38081658]
 [ 0.45429515 -0.79810125  0.68542588  0.83346358]]


In [4]:
import torch
from torch import tensor

def attention(Xe,Wq,Wk,Wv,d_model):
    
    if Xe.shape[0] != Wq.shape[1] :
        raise Exception('dimension not correct')

    Q = np.matmul(Xe,Wq)

    if Xe.shape[0] != Wk.shape[1] :
            raise Exception('dimension not correct')

    K = np.matmul(Xe,Wk)

    if Xe.shape[0] != Wv.shape[1] :
            raise Exception('dimension not correct')

    V = np.matmul(Xe,Wv)

    Attention = np.matmul(Q,K.T)
    Attention = Attention / np.sqrt(d_model)
    Attention = tensor(Attention)
    softmax = Softmax(dim=-1)
    Attention = softmax(Attention)
    Attention = Attention.numpy()
    #Attention = np.matmul(Attention , V)

    return Attention

In [5]:
A = attention(Xe,Wq,Wk,Wv,d_model)
print(A)


[[0.33680024 0.28656544 0.17242619 0.20420812]
 [0.32889729 0.28362755 0.17766758 0.20980758]
 [0.33787218 0.28871951 0.1805666  0.19284171]
 [0.33187172 0.28667232 0.18029939 0.20115657]]


In [6]:
WO = np.random.rand(d_model , d_model)
def layer_normalisation(X):
    epsilon = 10**-4
    mean = np.mean(X)
    std = np.std(X)
    return (X - mean) / (np.sqrt((std**2 + epsilon) ) )

def multi_head_projection(V , WO):
    Z = np.matmul(A,V)
    Y_att = np.matmul(Z,WO)
    return  Y_att

def first_risdual_addition(X_in,V,WO):
      Y_attn = multi_head_projection(V,WO)
      X_1 = X_in + Y_attn
      X_1 = layer_normalisation(d_model,d_model)
      return X_1

def postion_wise_feed_forward(X,W1,W2,bias1,bias2):
     X = np.matmul(X,W1) + bias1
     tensor_X = torch.tensor(X)
     tensor_X = torch.nn.functional.relu(tensor_X)
     X = tensor_X.numpy()
     Y_ffn = np.matmul(X,W2) + bias2
     return Y_ffn


def Encoder_output(Y_ffn):
     Y_ffn = X_embeding + Y_ffn
     H_encoder = layer_normalisation(Y_ffn)
     return H_encoder


def Encoder_operations(dff):

    # the dff is changing the dimension of projection then reproject it into the orginal projection dimension
     V = np.matmul(Xe,Wv)
     bias1 = np.random.rand(d_model)
     bias2 = np.random.rand(d_model)
     W1 = np.random.rand(d_model,dff)
     W2 = np.random.rand(dff,d_model)
     Y_att = multi_head_projection(V,WO)
     X_1 = layer_normalisation(Xe) + layer_normalisation(Y_att)
     Y_ffn = postion_wise_feed_forward(X_1,W1,W2,bias1,bias2)
     H_encoder = Encoder_output(Y_ffn)
     return H_encoder


H_encoder = Encoder_output(3)
print(H_encoder)





     
     
     



[[ 0.42908244  1.6288489  -0.72113744 -0.17467859]
 [ 1.65995555  0.96755683 -0.84332546 -0.26598471]
 [ 0.99713975 -1.39124841  0.4614209  -0.44892441]
 [-0.34277252 -2.15206547 -0.0088661   0.20499875]]
